In [ ]:
"""
Car price prediction — cleaned & fixed training pipeline
==========================================================
This fixes the root cause of "test accuracy > train accuracy" found in the
original notebook (see explanation below) and trains solid regression models.

WHAT WAS WRONG IN THE ORIGINAL NOTEBOOK
----------------------------------------
1. Data error, not model overfitting: one row (a used Mazda 323) had
   Price_numeric = 3,450 -- versus a dataset average of ~9.5 million. That's
   almost certainly a scraping/parsing error, not a real price.
2. The "Accuracy" metric was (1 - MAPE) * 100. MAPE divides each error by the
   ACTUAL value, so a near-zero actual price makes the % error explode
   (in this case, over 100,000% for that single row).
3. Because train_test_split randomly assigns ~70% of rows to train, this one
   corrupted row happened to land in the training set (confirmed below), not
   the test set. That single row was enough to drag the training "accuracy"
   score down by tens of percentage points, while the test set had no such
   row and looked artificially better.
4. Note this ONLY affected the MAPE-based "Accuracy" print. The R² and RMSE
   numbers in the notebook already showed the normal pattern (train fitting
   better than test) -- e.g. XGBoost train R²=0.98 vs test R²=0.88. That's
   the real, correct signal; the MAPE "accuracy" number was the misleading one.

Additional issues fixed here:
- Only 'Mileage_numeric' was scaled; 'YOM', 'Engine (cc)', 'Number of
  Cylinders' were passed through unscaled. That's fine for tree models but
  breaks distance/gradient-based models like SGD and SVR (which is why SGD
  blew up to a nonsensical negative-billions R² in the original run).
- Price ranges from ~Rs.170,000 to Rs.65,000,000 -- extremely right-skewed.
  Regressing on log(price) instead of raw price stabilizes this for every
  model, especially linear ones.
- 'YOM' (year of manufacture) was used directly; 'Car_Age' generalizes better
  to future data than a raw calendar year.
- Mean-based MAPE ("Accuracy") is fragile to outliers by construction. This
  script keeps R² / RMSE / MAE as the primary metrics and adds Median APE
  (MdAPE), which is far more robust, instead of mean MAPE.

Run:  pip install pandas numpy scikit-learn xgboost joblib --break-system-packages
      python train_car_price_model.py
"""

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor
import joblib

RANDOM_STATE = 42
CURRENT_YEAR = 2026
DATA_PATH = "formatted_data.csv"
PRICE_FLOOR = 50_000   # anything below this for a registered used car is a data error, not a real price

# ---------------------------------------------------------------------------
# 1. Load & clean
# ---------------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)

df = df.drop_duplicates()
df = df.dropna(subset=["Price_numeric", "Mileage_numeric", "Number of Cylinders", "Engine (cc)", "YOM"])
df = df[(df["Engine (cc)"] > 0) & (df["Engine (cc)"] < 10_000)]
df = df[df["Condition"] != "Antique"]

n_before = len(df)
df = df[df["Price_numeric"] >= PRICE_FLOOR]
print(f"Dropped {n_before - len(df)} row(s) with implausible Price_numeric < Rs.{PRICE_FLOOR:,} "
      f"(these are almost certainly data-entry/scraping errors, e.g. Price_numeric=3,450)")

# keep models with >=2 records, and (model, condition) groups with >=3 records,
# so a stratified split is possible
model_counts = df["Model_cleaned"].value_counts()
df = df[df["Model_cleaned"].isin(model_counts[model_counts >= 2].index)]

df["strat_col"] = list(zip(df["Model_cleaned"], df["Condition"]))
strat_counts = df["strat_col"].value_counts()
df = df[df["strat_col"].isin(strat_counts[strat_counts >= 3].index)]

print(f"Rows after cleaning: {len(df)}")

# ---------------------------------------------------------------------------
# 2. Feature engineering
# ---------------------------------------------------------------------------
df["Car_Age"] = CURRENT_YEAR - df["YOM"]

for col in ["Power Steering", "Power Shutters", "Power Mirrors"]:
    df[col] = df[col].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0}).fillna(0)

numeric_features = ["Car_Age", "Mileage_numeric", "Engine (cc)", "Number of Cylinders",
                     "Power Steering", "Power Shutters", "Power Mirrors"]
categorical_features = ["Brand", "Model_cleaned", "Gear", "Fuel Type", "Condition", "Body Type"]

X = df[numeric_features + categorical_features]
y_log = np.log1p(df["Price_numeric"])          # log-transform: price is heavily right-skewed

# ---------------------------------------------------------------------------
# 3. Train/test split (stratified by model+condition, like the original)
# ---------------------------------------------------------------------------
X_train, X_test, y_train, y_test, price_train, price_test = train_test_split(
    X, y_log, df["Price_numeric"],
    test_size=0.2, random_state=RANDOM_STATE, stratify=df["strat_col"]
)
print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")

# ---------------------------------------------------------------------------
# 4. Preprocessing -- ALL numeric columns are scaled consistently this time
# ---------------------------------------------------------------------------
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
])

# ---------------------------------------------------------------------------
# 5. Models + hyperparameter search
# ---------------------------------------------------------------------------
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

model_configs = {
    "Ridge (baseline)": (
        Ridge(),
        {"model__alpha": [0.1, 1, 10, 50, 100]},
    ),
    "Random Forest": (
        RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        {
            "model__n_estimators": [200, 300, 400],
            "model__max_depth": [None, 15, 25, 35],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": [0.5, "sqrt"],
        },
    ),
    "XGBoost": (
        XGBRegressor(objective="reg:squarederror", random_state=RANDOM_STATE, n_jobs=-1),
        {
            "model__n_estimators": [300, 500, 700],
            "model__max_depth": [3, 4, 5, 6],
            "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
            "model__subsample": [0.7, 0.85, 1.0],
            "model__colsample_bytree": [0.7, 0.85, 1.0],
        },
    ),
}


def evaluate(model, X_eval, price_true, label):
    """Evaluate on the ORIGINAL price scale (inverse of log1p), using metrics
    that are not distorted by near-zero actual values."""
    pred_price = np.clip(np.expm1(model.predict(X_eval)), 0, None)
    mae = mean_absolute_error(price_true, pred_price)
    rmse = np.sqrt(mean_squared_error(price_true, pred_price))
    r2 = r2_score(price_true, pred_price)
    ape = np.abs(price_true.values - pred_price) / price_true.values
    mdape = np.median(ape) * 100      # median, not mean -> robust to outliers
    print(f"  {label:5s} | R2: {r2:.4f} | RMSE: {rmse:>13,.0f} | MAE: {mae:>13,.0f} | Median APE: {mdape:5.2f}%")
    return {"r2": r2, "rmse": rmse, "mae": mae, "mdape": mdape}


results, best_estimators = {}, {}

for name, (estimator, params) in model_configs.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
    search = RandomizedSearchCV(
        pipe, params, n_iter=25, cv=cv,
        scoring="neg_root_mean_squared_error",
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    search.fit(X_train, y_train)
    best_estimators[name] = search.best_estimator_

    print(f"\n{name}  (best params: {search.best_params_})")
    train_m = evaluate(search.best_estimator_, X_train, price_train, "TRAIN")
    test_m = evaluate(search.best_estimator_, X_test, price_test, "TEST")
    results[name] = {"train": train_m, "test": test_m}

# ---------------------------------------------------------------------------
# 6. Summary + save best model
# ---------------------------------------------------------------------------
summary = pd.DataFrame({
    name: {
        "Train R2": r["train"]["r2"], "Test R2": r["test"]["r2"],
        "Train RMSE": r["train"]["rmse"], "Test RMSE": r["test"]["rmse"],
        "Train MdAPE%": r["train"]["mdape"], "Test MdAPE%": r["test"]["mdape"],
    }
    for name, r in results.items()
}).T.sort_values("Test RMSE")

print("\n" + "=" * 90)
print("MODEL COMPARISON (sorted by Test RMSE)")
print("=" * 90)
print(summary.to_string(float_format=lambda x: f"{x:,.4f}"))

best_name = summary.index[0]
best_model = best_estimators[best_name]
joblib.dump(best_model, "best_car_price_model.joblib")
print(f"\nBest model: {best_name} -> saved to best_car_price_model.joblib")

if hasattr(best_model.named_steps["model"], "feature_importances_"):
    feat_names = best_model.named_steps["preprocessor"].get_feature_names_out()
    importances = best_model.named_steps["model"].feature_importances_
    top = pd.Series(importances, index=feat_names).sort_values(ascending=False).head(15)
    print("\nTop 15 feature importances:")
    print(top.to_string(float_format=lambda x: f"{x:.4f}"))

Dropped 1 row(s) with implausible Price_numeric < Rs.50,000 (these are almost certainly data-entry/scraping errors, e.g. Price_numeric=3,450)
Rows after cleaning: 4002
Train rows: 3201, Test rows: 801


C:\Users\savin\AppData\Roaming\Python\Python314\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 5 is smaller than n_iter=25. Running 5 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(



Ridge (baseline)  (best params: {'model__alpha': 1})
  TRAIN | R2: 0.8872 | RMSE:     1,899,425 | MAE:       823,404 | Median APE:  5.66%
  TEST  | R2: 0.9180 | RMSE:     1,512,863 | MAE:       834,272 | Median APE:  6.31%

Random Forest  (best params: {'model__n_estimators': 300, 'model__min_samples_leaf': 2, 'model__max_features': 0.5, 'model__max_depth': 15})
  TRAIN | R2: 0.9291 | RMSE:     1,506,330 | MAE:       542,860 | Median APE:  3.33%
  TEST  | R2: 0.9272 | RMSE:     1,425,506 | MAE:       670,636 | Median APE:  4.20%

XGBoost  (best params: {'model__subsample': 0.7, 'model__n_estimators': 500, 'model__max_depth': 6, 'model__learning_rate': 0.01, 'model__colsample_bytree': 0.7})
  TRAIN | R2: 0.9146 | RMSE:     1,652,996 | MAE:       747,849 | Median APE:  5.42%
  TEST  | R2: 0.9211 | RMSE:     1,484,365 | MAE:       775,789 | Median APE:  5.58%

MODEL COMPARISON (sorted by Test RMSE)
                  Train R2  Test R2     Train RMSE      Test RMSE  Train MdAPE%  Test MdAP